# ERA5 Pressure-Level Monthly Means — South Asia

**ENSO / monsoon forecasting project** — extract ERA5 monthly-mean pressure-level
fields over **South Asia (5°N–40°N, 60°E–100°E)** on a **2°×2°** grid, save a CSV +
JSON metadata, and upload to a dedicated Google Drive folder.

| Field | ERA5 variable | Level | Notes |
|-------|---------------|-------|-------|
| `u850`   | `u_component_of_wind` | 850 hPa | zonal wind (m/s) |
| `v850`   | `v_component_of_wind` | 850 hPa | meridional wind (m/s) |
| `gph500` | `geopotential` | 500 hPa | geopotential height = geopotential / 9.80665 (m) |
| `d200`   | `divergence` | 200 hPa | upper-level divergence (s⁻¹) |

> **Note on velocity potential:** the requested *200 hPa velocity potential* is **not**
> a downloadable variable in `reanalysis-era5-pressure-levels-monthly-means`. Velocity
> potential χ is a derived field (∇²χ = δ) requiring a global spherical-harmonic
> inversion. As agreed, we use **200 hPa divergence** (δ) directly instead.

**Dataset:** `reanalysis-era5-pressure-levels-monthly-means` ·
[CDS page](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-pressure-levels-monthly-means)

## 1. Imports & configuration

In [1]:
import os
import json
import time

import numpy as np
import pandas as pd
import xarray as xr
import cdsapi

# --- Local output folder (separate from the existing ERA5 single-levels data) ---
DATA_DIR = "../../data/ERA5_pressure_levels"
os.makedirs(DATA_DIR, exist_ok=True)

# --- CDS dataset ---
DATASET = "reanalysis-era5-pressure-levels-monthly-means"

# --- Region: South Asia (5N-40N, 60E-100E) as area = [North, West, South, East] ---
AREA = [40, 60, 5, 100]
GRID = [2.0, 2.0]

# --- Time range (monthly means) ---
START_YEAR = 1980
END_YEAR = 2025

# --- Fields to download (one CDS request per field for unambiguous handling) ---
FIELDS = [
    {"name": "u850", "variable": "u_component_of_wind", "level": "850", "units": "m s-1",
     "long_name": "850 hPa u-component of wind"},
    {"name": "v850", "variable": "v_component_of_wind", "level": "850", "units": "m s-1",
     "long_name": "850 hPa v-component of wind"},
    {"name": "z500", "variable": "geopotential", "level": "500", "units": "m2 s-2",
     "long_name": "500 hPa geopotential"},
    {"name": "d200", "variable": "divergence", "level": "200", "units": "s-1",
     "long_name": "200 hPa divergence"},
]

# Standard gravity, for geopotential -> geopotential height (m)
G0 = 9.80665

print(f"Output dir : {DATA_DIR}")
print(f"Dataset    : {DATASET}")
print(f"Region     : N{AREA[0]} W{AREA[1]} S{AREA[2]} E{AREA[3]}  grid {GRID}")
print(f"Time range : {START_YEAR}-{END_YEAR} monthly")

Output dir : ../../data/ERA5_pressure_levels
Dataset    : reanalysis-era5-pressure-levels-monthly-means
Region     : N40 W60 S5 E100  grid [2.0, 2.0]
Time range : 1980-2025 monthly


## 2. Download each field from CDS\n\nOne request per (variable, level) so every NetCDF holds a single field. Existing\nvalid files are reused, and each retrieve is retried up to 3 times on transient errors.

In [2]:
def download_field(client, field, retries=3, wait=60):
    """Download one ERA5 pressure-level field as a NetCDF; reuse if already present."""
    path = os.path.join(DATA_DIR, f"era5_{field['name']}_{START_YEAR}_{END_YEAR}.nc")
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"  {field['name']}: already downloaded -> {os.path.basename(path)}")
        return path

    request = {
        "product_type": ["monthly_averaged_reanalysis"],
        "variable": [field["variable"]],
        "pressure_level": [field["level"]],
        "year": [str(y) for y in range(START_YEAR, END_YEAR + 1)],
        "month": [f"{m:02d}" for m in range(1, 13)],
        "time": ["00:00"],
        "area": AREA,           # [N, W, S, E]
        "grid": GRID,           # 2x2 degrees
        "data_format": "netcdf",
        "download_format": "unarchived",
    }

    for attempt in range(1, retries + 1):
        try:
            client.retrieve(DATASET, request, path)
            print(f"  {field['name']}: downloaded -> {os.path.basename(path)}")
            return path
        except Exception as e:
            if attempt == retries:
                raise
            print(f"  {field['name']}: retrieve failed (attempt {attempt}/{retries}): {e}")
            print(f"  retrying in {wait}s ...")
            time.sleep(wait)


client = cdsapi.Client()

paths = {}
for field in FIELDS:
    paths[field["name"]] = download_field(client, field)

print("\nAll fields downloaded.")

2026-06-27 14:07:20,835 INFO Request ID is 7a060116-188d-4c78-aec3-a092813cf074
2026-06-27 14:07:21,382 INFO status has been updated to accepted
2026-06-27 14:07:45,215 INFO status has been updated to running
2026-06-27 14:08:40,046 INFO status has been updated to successful


  u850: downloaded -> era5_u850_1980_2025.nc


2026-06-27 14:08:45,246 INFO Request ID is fc976b09-a2d6-4c2a-81d0-009cf259c44a
2026-06-27 14:08:45,770 INFO status has been updated to accepted
2026-06-27 14:09:00,654 INFO status has been updated to running
2026-06-27 14:10:05,539 INFO status has been updated to successful


  v850: downloaded -> era5_v850_1980_2025.nc


2026-06-27 14:10:13,519 INFO Request ID is 095af165-f1c7-41b4-813c-6bb96526e64c
2026-06-27 14:10:13,757 INFO status has been updated to accepted
2026-06-27 14:10:32,707 INFO status has been updated to running
2026-06-27 14:11:39,008 INFO status has been updated to successful


  z500: downloaded -> era5_z500_1980_2025.nc


2026-06-27 14:11:46,262 INFO Request ID is 1de39bfa-7caf-4783-bce7-e21ab3fc4bd0
2026-06-27 14:11:46,489 INFO status has been updated to accepted
2026-06-27 14:12:45,827 INFO status has been updated to running
Recovering from connection error [HTTPSConnectionPool(host='cds.climate.copernicus.eu', port=443): Read timed out. (read timeout=60)], attempt 1 of 500
Retrying in 120 seconds
2026-06-27 14:15:52,666 INFO status has been updated to successful
                                                                                      

  d200: downloaded -> era5_d200_1980_2025.nc

All fields downloaded.


## 3. Merge fields, derive geopotential height, save CSV + metadata\n\nEach single-level NetCDF is opened, the pressure-level dimension squeezed away, and\nthe fields merged on (time, latitude, longitude). 500 hPa geopotential is converted\nto geopotential height, and the divergence field is kept as the upper-level field.

In [3]:
def open_field(path, name):
    """Open a single-field ERA5 NetCDF, drop the pressure-level dim, return a named DataArray."""
    ds = xr.open_dataset(path)

    # Normalise the time coordinate name (newer CDS files use 'valid_time')
    if "valid_time" in ds.coords and "time" not in ds.coords:
        ds = ds.rename({"valid_time": "time"})

    # The geophysical variable is the one defined on latitude & longitude
    main = [v for v in ds.data_vars if {"latitude", "longitude"} <= set(ds[v].dims)][0]

    da = ds[main].squeeze(drop=True)        # drop the size-1 pressure-level dimension
    da = da.reset_coords(drop=True)         # drop leftover scalar coords (pressure_level, number, expver)
    return da.rename(name)


# Open and merge all four fields
das = [open_field(paths[f["name"]], f["name"]) for f in FIELDS]
ds_merged = xr.merge(das, compat="override", join="outer")

# Convert 500 hPa geopotential -> geopotential height (m), then drop the raw field
ds_merged["gph500"] = ds_merged["z500"] / G0
ds_merged["gph500"].attrs = {"units": "m", "long_name": "500 hPa geopotential height"}
ds_merged = ds_merged.drop_vars("z500")

# Make sure the time coordinate is datetime
ds_merged = ds_merged.sortby("time")

# Long-format DataFrame with a tidy column order
col_order = ["time", "latitude", "longitude", "u850", "v850", "gph500", "d200"]
df = ds_merged.to_dataframe().reset_index()
df = df[[c for c in col_order if c in df.columns]]

csv_path = os.path.join(DATA_DIR, f"era5_pressure_levels_{START_YEAR}_{END_YEAR}.csv")
df.to_csv(csv_path, index=False)
print(f"CSV saved: {csv_path} ({len(df)} rows)")

# --- Metadata JSON ---
times = pd.to_datetime(ds_merged.time.values)
field_units = {
    "u850": "m s-1",
    "v850": "m s-1",
    "gph500": "m",
    "d200": "s-1",
}
field_levels = {"u850": 850, "v850": 850, "gph500": 500, "d200": 200}
metadata = {
    "dataset": DATASET,
    "region": {"name": "South Asia", "north": AREA[0], "west": AREA[1], "south": AREA[2], "east": AREA[3]},
    "grid": {"resolution_deg": GRID,
              "n_lat": int(ds_merged.sizes["latitude"]),
              "n_lon": int(ds_merged.sizes["longitude"])},
    "latitude_range": [float(ds_merged.latitude.min()), float(ds_merged.latitude.max())],
    "longitude_range": [float(ds_merged.longitude.min()), float(ds_merged.longitude.max())],
    "time_range": [str(times[0]), str(times[-1])],
    "n_time": int(ds_merged.sizes["time"]),
    "dimensions": {k: int(v) for k, v in ds_merged.sizes.items()},
    "variables": {
        name: {
            "units": field_units[name],
            "pressure_level_hPa": field_levels[name],
            "shape": list(ds_merged[name].shape),
        }
        for name in ["u850", "v850", "gph500", "d200"]
    },
    "notes": "200 hPa divergence used in place of velocity potential (not a downloadable ERA5 variable).",
    "csv_rows": int(len(df)),
}
meta_path = os.path.join(DATA_DIR, f"era5_pressure_levels_{START_YEAR}_{END_YEAR}_metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"Metadata saved: {meta_path}")

CSV saved: ../../data/ERA5_pressure_levels/era5_pressure_levels_1980_2025.csv (208656 rows)
Metadata saved: ../../data/ERA5_pressure_levels/era5_pressure_levels_1980_2025_metadata.json


## 4. Upload to Google Drive (dedicated folder)\n\nUpload only the two derived files (CSV + metadata) into a new\n`ERA5_pressure_levels` folder under the shared Drive parent.

In [4]:
import pickle
import mimetypes
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

SCOPES = ["https://www.googleapis.com/auth/drive"]
TOKEN_PATH = "../../token.pickle"
CLIENT_SECRETS = "../../client_secrets.json"
DRIVE_PARENT_ID = "1zLJgkYkrM1LRgtl7v5sfAQ4aits9zfUq"

# Dedicated Drive folder for this dataset
DRIVE_FOLDER_NAME = "ERA5_pressure_levels"

FILES_TO_UPLOAD = [csv_path, meta_path]


def get_drive_service():
    creds = None
    if os.path.exists(TOKEN_PATH):
        with open(TOKEN_PATH, "rb") as f:
            creds = pickle.load(f)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRETS, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(TOKEN_PATH, "wb") as f:
            pickle.dump(creds, f)
    return build("drive", "v3", credentials=creds)


def get_or_create_folder(service, name, parent_id):
    query = (
        f"name = '{name}' and mimeType = 'application/vnd.google-apps.folder' "
        f"and '{parent_id}' in parents and trashed = false"
    )
    resp = service.files().list(q=query, spaces="drive", fields="files(id, name)").execute()
    files = resp.get("files", [])
    if files:
        return files[0]["id"]
    folder = service.files().create(
        body={"name": name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]},
        fields="id",
    ).execute()
    return folder["id"]


def upload_file(service, path, folder_id):
    name = os.path.basename(path)
    mimetype = mimetypes.guess_type(path)[0] or "application/octet-stream"
    media = MediaFileUpload(path, mimetype=mimetype, resumable=True)
    request = service.files().create(
        body={"name": name, "parents": [folder_id]},
        media_body=media,
        fields="id, name",
    )
    response = None
    while response is None:
        _, response = request.next_chunk()
    return response["id"]


service = get_drive_service()
folder_id = get_or_create_folder(service, DRIVE_FOLDER_NAME, DRIVE_PARENT_ID)
print(f"Drive folder: {DRIVE_FOLDER_NAME} (id: {folder_id})")

uploaded = {}
for path in FILES_TO_UPLOAD:
    fid = upload_file(service, path, folder_id)
    uploaded[os.path.basename(path)] = fid
    print(f"  Uploaded: {os.path.basename(path)} (id: {fid})")

print("Done.")

Drive folder: ERA5_pressure_levels (id: 1HxLZmmSzI4_K0mxLM_DKWxx2duK14jak)
  Uploaded: era5_pressure_levels_1980_2025.csv (id: 1xikDxR1YCNEkdmTG4Bo3qrTkKMCRzGfi)
  Uploaded: era5_pressure_levels_1980_2025_metadata.json (id: 1GcFKFf7oNln1Ss_fbpA_NYP8d6K6CKY4)
Done.


## 5. Validation

In [5]:
# Grid / coverage checks
lat = np.sort(df["latitude"].unique())
lon = np.sort(df["longitude"].unique())
n_time = df["time"].nunique()
expected_rows = n_time * lat.size * lon.size

print("--- Validation ---")
print(f"Columns        : {list(df.columns)}")
print(f"Latitudes      : {lat.min()}..{lat.max()} step {np.unique(np.diff(lat))} ({lat.size} pts)")
print(f"Longitudes     : {lon.min()}..{lon.max()} step {np.unique(np.diff(lon))} ({lon.size} pts)")
print(f"Time steps     : {n_time} ({df['time'].min()} .. {df['time'].max()})")
print(f"CSV rows       : {len(df)} (expected {n_time} x {lat.size} x {lon.size} = {expected_rows})")

print("\nField sanity ranges:")
for name, unit in [("u850", "m/s"), ("v850", "m/s"), ("gph500", "m"), ("d200", "1/s")]:
    print(f"  {name:7s} min/max: {df[name].min():.4g} .. {df[name].max():.4g} {unit}")

print(f"\nDrive folder '{DRIVE_FOLDER_NAME}' uploads: {list(uploaded.keys())}")

--- Validation ---
Columns        : ['time', 'latitude', 'longitude', 'u850', 'v850', 'gph500', 'd200']
Latitudes      : 5.0..39.0 step [2.] (18 pts)
Longitudes     : 60.0..100.0 step [2.] (21 pts)
Time steps     : 552 (1980-01-01 00:00:00 .. 2025-12-01 00:00:00)
CSV rows       : 208656 (expected 552 x 18 x 21 = 208656)

Field sanity ranges:
  u850    min/max: -12.41 .. 25.86 m/s
  v850    min/max: -13.82 .. 10.97 m/s
  gph500  min/max: 5477 .. 5911 m
  d200    min/max: -8.695e-05 .. 4.926e-05 1/s

Drive folder 'ERA5_pressure_levels' uploads: ['era5_pressure_levels_1980_2025.csv', 'era5_pressure_levels_1980_2025_metadata.json']
